# Probabilidades de fútbol internacional
Análisis local de XGBoost, Poisson y Dixon-Coles. Ejecutar primero `python run_forecast.py` desde la raíz. Este notebook carga resultados; no reentrena automáticamente.

In [ ]:
from pathlib import Path
import json
import pandas as pd
if 'display' not in globals():
    display = print
ROOT = Path.cwd() if (Path.cwd() / 'reports').exists() else Path.cwd().parent
OUT = ROOT / 'reports' / 'probabilistic'
audit = json.loads((OUT / 'data_audit.json').read_text(encoding='utf-8'))
{key: audit[key] for key in ['prediction_cutoff_exclusive', 'latest_scored_date', 'selected_score_model', 'elapsed_seconds', 'simulations']}

## Comparación temporal
El modelo de marcadores se elige en 2022 por NLL. La prueba posterior conserva parámetros congelados y variables actualizadas solo con resultados anteriores. Las métricas no se atribuyen a los modelos después reentrenados para el fixture.

In [ ]:
comparison = pd.read_csv(OUT / 'model_comparison.csv')
display(comparison[comparison['split'] == 'test'].sort_values('log_loss'))

![Calibración temporal](../reports/probabilistic/calibration.png)

## Probabilidades por partido
Las probabilidades dependen del rival y la sede. La escasez de etiquetas de empate por argmax no demuestra por sí sola mala calibración.

In [ ]:
matches = pd.read_csv(OUT / 'match_probabilities.csv')
selected = matches[matches['model'] == audit['selected_score_model']]
display(selected[['home_team', 'away_team', 'p_home_win', 'p_draw', 'p_away_win', 'top_scorelines']])
assert len(selected) == 72
assert ((selected[['p_home_win', 'p_draw', 'p_away_win']].sum(axis=1) - 1).abs() < 1e-8).all()

## Marcadores e interpretación
Los cinco marcadores principales no agotan la distribución. La importancia por permutación es un diagnóstico con una muestra pequeña, no evidencia causal. Los ZIP locales contienen fixtures 2026 sin resultados; no permiten evaluar aciertos del Mundial 2026.

In [ ]:
scores = pd.read_csv(OUT / 'scoreline_probabilities.csv')
match_id = int(selected.iloc[0]['match_id'])
block = scores[(scores['model'] == audit['selected_score_model']) & (scores['match_id'] == match_id)]
display(block.sort_values('probability', ascending=False).head(10))
assert abs(block['probability'].sum() - 1) < 1e-8
display(pd.read_csv(OUT / 'permutation_importance.csv').sort_values('log_loss_increase', ascending=False))